# Setup de Librerias

In [10]:
# For calculation
import numpy as np
import os

# For visualization
import matplotlib.pyplot as plt
from skimage import io, color, filters, measure
import seaborn as sns

# Funciones a Utilizar

Image Reading

In [11]:
def num2fixstr(x,d):
    # input: (21,5)
    # output: '00021'
    st = '%0*d' % (d,x)
    return st

def build_filename(prefix,num_class,digits_class,num_img,digits_img,sep='_'):
  return prefix + num2fixstr(num_class,digits_class) + sep + num2fixstr(num_img,digits_img) + '.png'

def imageload(prefix,num_class,digits_class,num_img,digits_img,sep='_',echo='off'):
  st   = build_filename(prefix,num_class,digits_class,num_img,digits_img,sep)
  if echo == 'on':
    print('loading image '+st+'...')
  img    = plt.imread(st)
  return img

Limpieza de ruido: conservar el blob de mayor área

In [12]:
def keep_largest_blob(binary):
    """Remove noise from a binary image, keeping only the largest connected blob.

    Labels every connected component in the binary image, measures each
    component's area, and returns a binary mask containing only the
    component with the largest area.

    Parameters
    ----------
    binary : ndarray (bool)
        The segmented (binarized) image, e.g. the output of `segment_otsu`.

    Returns
    -------
    cleaned : ndarray (bool)
        Binary mask with only the largest-area blob retained.
    """
    labels = measure.label(binary)

    if labels.max() == 0:
        return binary

    areas = np.bincount(labels.ravel())
    areas[0] = 0  # ignore the background label
    largest_label = np.argmax(areas)

    return labels == largest_label

Segmentación de Otsu

In [13]:
def segment_otsu(image_path, invert=False):
    """Read an image and segment it using Otsu's thresholding method.

    After thresholding, only the largest connected blob is kept to
    remove noise (see `keep_largest_blob`).

    Parameters
    ----------
    image_path : str
        Path to the image file.
    invert : bool
        If True, foreground is treated as pixels BELOW the threshold
        (useful when letters are dark on a light background).

    Returns
    -------
    gray : ndarray
        The original image in grayscale.
    binary : ndarray (bool)
        The segmented (binarized) image, with only the largest blob retained.
    threshold : float
        The threshold value chosen automatically by Otsu's method.
    """
    img = io.imread(image_path)
    gray = color.rgb2gray(img) if img.ndim == 3 else img

    threshold = filters.threshold_otsu(gray)
    binary = gray < threshold if invert else gray > threshold
    binary = keep_largest_blob(binary)

    return gray, binary, threshold

Mostrar Segmentacion

In [14]:
def show_segmentation(image_path, invert=False):
    """Segment an image with Otsu's method and display original vs. result."""
    gray, binary, threshold = segment_otsu(image_path, invert=invert)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(gray, cmap="gray")
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(binary, cmap="gray")
    axes[1].set_title(f"Otsu segmentation (t={threshold:.3f})")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    return binary

Rellenar Agujeros

In [15]:
def fill_holes(binary_image):
    """
    Fill holes in a binary image.
    A hole is defined as a background region that is surrounded by foreground pixels.

    Parameters:
        binary_image (numpy.ndarray): A 2D binary NumPy array (0s and 1s).

    Returns:
        numpy.ndarray: Binary image with holes filled.
    """
    # Ensure input is binary (0 and 1 only)
    binary_image = (binary_image > 0).astype(np.uint8)

    # Create a padded version of the image to handle boundary cases
    padded = np.pad(binary_image, pad_width=1, mode='constant', constant_values=0)

    # Create a mask for background areas
    filled = np.zeros_like(padded, dtype=np.uint8)
    filled[0, 0] = 1  # Start flood-fill from the top-left corner

    # Structuring element for 4-connectivity (up, down, left, right)
    shifts = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    # Flood-fill algorithm
    while True:
        new_filled = filled.copy()
        for dx, dy in shifts:
            new_filled |= np.roll(filled, shift=dx, axis=0)
            new_filled |= np.roll(filled, shift=dy, axis=1)

        # Only allow propagation where the original padded image is background
        new_filled &= (padded == 0)

        if np.array_equal(filled, new_filled):
            break  # Stop when no new changes
        filled = new_filled

    # The holes are the complement of the flooded region inside the padded area
    holes_filled = np.where(filled[1:-1, 1:-1] == 0, 1, binary_image)

    return holes_filled

Area Normalizada

In [16]:
def NormalizedArea(S,echo='off'):

  if len(S.shape)>2:
    S = S[:,:,0]

  # If image has multiple channels (RGBA), use the first one for segmentation
  R = S>0                 # Binary image of the segmentation


  Rf = fill_holes(R).astype(int) # filled region (with no holes)

  # Area
  A  = np.sum(R)
  Af = np.sum(Rf)


  # Normalized area
  if Af == 0: # Avoid division by zero if width or height is 0
      a = 0.0
  else:
      a = A/Af

  if echo=='on':
    implot = plt.imshow(R,cmap='gray')
    print('normalized area = '+str(a))
  return a

Clasificación por área normalizada: P, @ vs. N, M, U

In [17]:
def classify_by_normalized_area(S, threshold=0.95):
    """Classify a segmented character as 'P o @' or 'N, M o U' by normalized area.

    P and @ each enclose a background region (a hole), so filling that hole
    increases their area and pushes the normalized area below 1. N, M and U
    have no holes, so their normalized area stays close to 1.

    Parameters
    ----------
    S : ndarray
        Segmented (binarized) character image, e.g. the output of `segment_otsu`.
    threshold : float
        Normalized area cutoff. Values below this are classified as 'P o @'.

    Returns
    -------
    str
        'P o @' if the normalized area is below `threshold`, otherwise 'N, M o U'.
    """
    a = NormalizedArea(S)
    return 'P o @' if a < threshold else 'N, M o U'

Excentricidad: diferenciar P de @

In [18]:
def Eccentricity(S, echo='off'):
  """Compute the eccentricity of a segmented character's filled region.

  Used to tell P from @: P is a tall, narrow stem-and-bowl shape (high
  eccentricity), while @ is close to circular (low eccentricity).

  Parameters
  ----------
  S : ndarray
      Segmented (binarized) character image, e.g. the output of `segment_otsu`.

  Returns
  -------
  float
      Eccentricity of the filled region's best-fit ellipse, in [0, 1).
      0 is a circle, values closer to 1 are more elongated.
  """
  if len(S.shape) > 2:
    S = S[:, :, 0]

  R = S > 0                      # Binary image of the segmentation
  Rf = fill_holes(R).astype(int) # filled region (with no holes)

  props = measure.regionprops(Rf)
  e = props[0].eccentricity if props else 0.0

  if echo == 'on':
    implot = plt.imshow(Rf, cmap='gray')
    print('eccentricity = ' + str(e))

  return e

# Entrenamiento

In [19]:
!mkdir seg_training

A subdirectory or file seg_training already exists.


Guardar imágenes segmentadas de entrenamiento

In [20]:
def save_training_segmentations(training_dir="training", char_prefix="char_",
                                 num_classes=5, digits_class=2,
                                 num_images=50, digits_img=3, sep='_',
                                 output_dir="seg_training", invert=False):
    """Segment every training image and save the results to `output_dir`.

    Filenames are built with the Image Reading section's naming convention
    (`build_filename`/`num2fixstr`), the same one `imageload` uses, so each
    output file keeps the same name as its corresponding input image:
    `seg_training/char_04_012.png` is the segmentation of
    `training/char_04_012.png`.

    Parameters
    ----------
    training_dir : str
        Folder containing the original training images.
    char_prefix : str
        Filename prefix before the zero-padded class number.
    num_classes : int
        Number of character classes (class numbers run 1..num_classes).
    digits_class : int
        Zero-padded width of the class number in the filename.
    num_images : int
        Number of images per class (image numbers run 1..num_images).
    digits_img : int
        Zero-padded width of the image number in the filename.
    sep : str
        Separator between the class and image numbers.
    output_dir : str
        Folder where segmented images are saved (created if it doesn't exist).
    invert : bool
        Passed through to `segment_otsu`.
    """
    os.makedirs(output_dir, exist_ok=True)

    for num_class in range(1, num_classes + 1):
        for num_img in range(1, num_images + 1):
            filename = build_filename(char_prefix, num_class, digits_class, num_img, digits_img, sep)

            image_path = os.path.join(training_dir, filename)
            _, binary, _ = segment_otsu(image_path, invert=invert)

            output_path = os.path.join(output_dir, filename)
            io.imsave(output_path, (binary * 255).astype(np.uint8))